# Understanding the Transformer: From Text to Prediction
### A hands-on walkthrough using Karpathy's nanochat

This notebook builds the transformer architecture from scratch, step by step, connecting every concept to the actual nanochat source code in `nanochat/gpt.py`.

**What we'll cover:**
1. Setup & device detection (MPS on M1)
2. Tokenization — text → token IDs
3. Token Embeddings — token IDs → vectors
4. Q, K, V Projections
5. Scaled Dot-Product Attention
6. Causal Masking
7. Multi-Head Attention
8. The MLP Block
9. Residual Connections & Layer Norm
10. A Full Transformer Block
11. Stacking Layers — the full GPT
12. Next Token Prediction
13. Running nanochat's actual model

**Prerequisites:** Run `uv sync --extra cpu && source .venv/bin/activate` from the nanochat directory, then launch this notebook.

---
## Architecture Overview

Here is the full pipeline we will build step by step in this notebook:

<div style="text-align:center; margin: 24px 0;">
<svg viewBox="0 0 620 1060" xmlns="http://www.w3.org/2000/svg" width="520" font-family="system-ui, sans-serif">
  <defs>
    <marker id="arr" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M0,0 L0,6 L8,3 z" fill="#888"/></marker>
    <marker id="arr-p" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M0,0 L0,6 L8,3 z" fill="#7c6af7"/></marker>
  </defs>
  <rect width="620" height="1060" fill="#fff"/>
  <!-- Title -->
  <text x="310" y="30" text-anchor="middle" font-size="14" font-weight="700" fill="#111">Transformer: Text &#x2192; Next Token</text>
  <!-- Step 1: Raw Text -->
  <rect x="185" y="44" width="250" height="50" rx="8" fill="#f0effe" stroke="#7c6af7" stroke-width="1.5"/>
  <text x="310" y="63" text-anchor="middle" font-size="10" font-weight="600" fill="#7c6af7">STEP 1</text>
  <text x="310" y="82" text-anchor="middle" font-size="12" font-weight="700" fill="#111">Raw Text</text>
  <text x="310" y="96" text-anchor="middle" font-size="9" fill="#888">"What is the capital of France?"</text>
  <line x1="310" y1="94" x2="310" y2="112" stroke="#888" stroke-width="1.5" marker-end="url(#arr)"/>
  <text x="326" y="106" font-size="8" fill="#888">BPE</text>
  <!-- Step 2: Tokenization -->
  <rect x="185" y="116" width="250" height="50" rx="8" fill="#f0effe" stroke="#7c6af7" stroke-width="1.5"/>
  <text x="310" y="135" text-anchor="middle" font-size="10" font-weight="600" fill="#7c6af7">STEP 2 &#x2014; Tokenization</text>
  <text x="310" y="153" text-anchor="middle" font-size="12" font-weight="700" fill="#111">Token IDs</text>
  <text x="310" y="167" text-anchor="middle" font-size="9" fill="#888">[2515, 374, 279, 6864, ...]</text>
  <line x1="310" y1="166" x2="310" y2="184" stroke="#888" stroke-width="1.5" marker-end="url(#arr)"/>
  <text x="322" y="178" font-size="8" fill="#888">wte lookup</text>
  <!-- Step 3: Embeddings -->
  <rect x="185" y="188" width="250" height="50" rx="8" fill="#f0effe" stroke="#7c6af7" stroke-width="1.5"/>
  <text x="310" y="207" text-anchor="middle" font-size="10" font-weight="600" fill="#7c6af7">STEP 3 &#x2014; Embedding</text>
  <text x="310" y="225" text-anchor="middle" font-size="12" font-weight="700" fill="#111">Token Vectors [B, T, n_embd]</text>
  <text x="310" y="239" text-anchor="middle" font-size="9" fill="#888">context-free</text>
  <line x1="310" y1="238" x2="310" y2="256" stroke="#888" stroke-width="1.5" marker-end="url(#arr)"/>
  <!-- Transformer Block box -->
  <rect x="55" y="260" width="510" height="406" rx="12" fill="none" stroke="#ccc" stroke-width="1.5" stroke-dasharray="6 4"/>
  <text x="82" y="278" font-size="9" font-weight="600" fill="#999">TRANSFORMER BLOCK  &#xD7;  N layers</text>
  <!-- Step 4: Q K V -->
  <rect x="185" y="284" width="250" height="56" rx="8" fill="#fff7ed" stroke="#f59e0b" stroke-width="1.5"/>
  <text x="310" y="303" text-anchor="middle" font-size="10" font-weight="600" fill="#f59e0b">STEP 4 &#x2014; Projections</text>
  <text x="310" y="321" text-anchor="middle" font-size="12" font-weight="700" fill="#111">Q, K, V Vectors</text>
  <text x="310" y="337" text-anchor="middle" font-size="9" fill="#888">x&#xB7;Wq &nbsp; x&#xB7;Wk &nbsp; x&#xB7;Wv</text>
  <line x1="310" y1="340" x2="310" y2="358" stroke="#888" stroke-width="1.5" marker-end="url(#arr)"/>
  <!-- Step 5+6: Attention -->
  <rect x="155" y="362" width="310" height="68" rx="8" fill="#fff7ed" stroke="#f59e0b" stroke-width="1.5"/>
  <text x="310" y="381" text-anchor="middle" font-size="10" font-weight="600" fill="#f59e0b">STEPS 5 + 6 &#x2014; Attention + Causal Mask</text>
  <text x="310" y="399" text-anchor="middle" font-size="12" font-weight="700" fill="#111">Scaled Dot-Product Attention</text>
  <text x="310" y="415" text-anchor="middle" font-size="9" fill="#888">softmax( Q&#xB7;K&#x1D40; / &#x221A;d ) &#xD7; V &nbsp;&#x2014;&nbsp; upper triangle &#x2192; &#x2212;&#x221E;</text>
  <line x1="310" y1="430" x2="310" y2="448" stroke="#888" stroke-width="1.5" marker-end="url(#arr)"/>
  <!-- Step 7: Multi-head -->
  <rect x="185" y="452" width="250" height="50" rx="8" fill="#fff7ed" stroke="#f59e0b" stroke-width="1.5"/>
  <text x="310" y="471" text-anchor="middle" font-size="10" font-weight="600" fill="#f59e0b">STEP 7 &#x2014; Multi-Head</text>
  <text x="310" y="489" text-anchor="middle" font-size="12" font-weight="700" fill="#111">Context Vectors</text>
  <text x="310" y="503" text-anchor="middle" font-size="9" fill="#888">H heads concat &#x2192; project &#x2192; [B,T,n_embd]</text>
  <!-- Residual left bypass -->
  <path d="M178,284 Q108,284 108,452 Q108,503 180,503" fill="none" stroke="#ccc" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#arr)"/>
  <text x="70" y="400" font-size="8" fill="#aaa" transform="rotate(-90,70,400)">residual (+x)</text>
  <line x1="310" y1="502" x2="310" y2="522" stroke="#888" stroke-width="1.5" marker-end="url(#arr)"/>
  <!-- Step 8: MLP -->
  <rect x="185" y="526" width="250" height="50" rx="8" fill="#fff7ed" stroke="#f59e0b" stroke-width="1.5"/>
  <text x="310" y="545" text-anchor="middle" font-size="10" font-weight="600" fill="#f59e0b">STEP 8 &#x2014; MLP</text>
  <text x="310" y="563" text-anchor="middle" font-size="12" font-weight="700" fill="#111">Feed-Forward Network</text>
  <text x="310" y="577" text-anchor="middle" font-size="9" fill="#888">expand 4&#xD7; &#x2192; relu&#xB2; &#x2192; contract</text>
  <!-- Residual right bypass -->
  <path d="M435,502 Q510,502 510,553 Q510,577 434,577" fill="none" stroke="#ccc" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#arr)"/>
  <text x="524" y="552" font-size="8" fill="#aaa" transform="rotate(90,524,552)">residual (+x)</text>
  <!-- Repeat loop -->
  <path d="M555,576 Q578,576 578,430 Q578,284 555,284" fill="none" stroke="#7c6af7" stroke-width="1.4" stroke-dasharray="5 3" marker-end="url(#arr-p)"/>
  <text x="592" y="436" font-size="8" fill="#7c6af7" transform="rotate(90,592,436)">repeat &#xD7; N layers</text>
  <!-- Step 9 note -->
  <text x="310" y="614" text-anchor="middle" font-size="8" fill="#aaa">RMSNorm before each sub-block (Steps 9 &#x2014; pre-norm)</text>
  <line x1="310" y1="666" x2="310" y2="682" stroke="#888" stroke-width="1.5" marker-end="url(#arr)"/>
  <!-- Step 11: LM Head -->
  <rect x="185" y="686" width="250" height="50" rx="8" fill="#f0effe" stroke="#7c6af7" stroke-width="1.5"/>
  <text x="310" y="705" text-anchor="middle" font-size="10" font-weight="600" fill="#7c6af7">STEP 11 &#x2014; LM Head</text>
  <text x="310" y="723" text-anchor="middle" font-size="12" font-weight="700" fill="#111">Logits [B, T, vocab_size]</text>
  <text x="310" y="737" text-anchor="middle" font-size="9" fill="#888">n_embd &#x2192; one score per vocabulary token</text>
  <line x1="310" y1="736" x2="310" y2="754" stroke="#888" stroke-width="1.5" marker-end="url(#arr)"/>
  <!-- Step 12: Sample -->
  <rect x="185" y="758" width="250" height="50" rx="8" fill="#f0effe" stroke="#7c6af7" stroke-width="1.5"/>
  <text x="310" y="777" text-anchor="middle" font-size="10" font-weight="600" fill="#7c6af7">STEP 12 &#x2014; Sample</text>
  <text x="310" y="795" text-anchor="middle" font-size="12" font-weight="700" fill="#111">Next Token</text>
  <text x="310" y="809" text-anchor="middle" font-size="9" fill="#888">softmax &#x2192; probabilities &#x2192; sample</text>
  <line x1="310" y1="808" x2="310" y2="826" stroke="#888" stroke-width="1.5" marker-end="url(#arr)"/>
  <!-- Output -->
  <rect x="210" y="830" width="200" height="42" rx="8" fill="#7c6af7"/>
  <text x="310" y="847" text-anchor="middle" font-size="10" font-weight="600" fill="#fff">Output</text>
  <text x="310" y="864" text-anchor="middle" font-size="14" font-weight="700" fill="#fff">"Paris"</text>
  <!-- Autoregressive loop -->
  <path d="M410,851 Q536,851 536,68 Q536,44 435,44" fill="none" stroke="#7c6af7" stroke-width="1.4" stroke-dasharray="5 3" marker-end="url(#arr-p)"/>
  <text x="550" y="448" font-size="8" fill="#7c6af7" transform="rotate(90,550,448)">append token &#x2192; repeat (autoregressive)</text>
  <!-- KV Cache note -->
  <rect x="55" y="888" width="510" height="30" rx="6" fill="#f7f7f8"/>
  <text x="310" y="905" text-anchor="middle" font-size="8" font-weight="600" fill="#888">KV CACHE (inference) &#x2014; K and V for past tokens are stored; only the new token is recomputed</text>
  <!-- Legend -->
  <rect x="55" y="928" width="510" height="30" rx="6" fill="#f7f7f8"/>
  <rect x="72" y="938" width="10" height="10" rx="2" fill="#f0effe" stroke="#7c6af7" stroke-width="1.2"/>
  <text x="87" y="947" font-size="9" fill="#333">Input / Output</text>
  <rect x="185" y="938" width="10" height="10" rx="2" fill="#fff7ed" stroke="#f59e0b" stroke-width="1.2"/>
  <text x="200" y="947" font-size="9" fill="#333">Inside block</text>
  <line x1="308" y1="943" x2="326" y2="943" stroke="#ccc" stroke-width="1.4" stroke-dasharray="4 3"/>
  <text x="330" y="947" font-size="9" fill="#333">Residual</text>
  <line x1="408" y1="943" x2="426" y2="943" stroke="#7c6af7" stroke-width="1.4" stroke-dasharray="5 3"/>
  <text x="430" y="947" font-size="9" fill="#333">Loop</text>
  <!-- Source note -->
  <text x="310" y="990" text-anchor="middle" font-size="8" fill="#bbb">nanochat/gpt.py &#x2014; CausalSelfAttention &#x2014; Block &#x2014; GPT.forward</text>
</svg>
</div>

---
## Step 0: Setup

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ── Device detection ──────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')   # Apple Silicon GPU
else:
    device = torch.device('cpu')

print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

Using device: mps
PyTorch version: 2.9.1


---
## Step 1: Tokenization — Text → Token IDs

Before the transformer sees anything, text is split into **tokens** — small chunks of characters. Each token is mapped to an integer ID.

nanochat trains its own BPE tokenizer (see `scripts/tok_train.py`). For this notebook we use `tiktoken` (GPT-4 style) which is already installed as a dependency and works the same way.

In [2]:
import tiktoken

# GPT-4 tokenizer (same BPE style as nanochat's custom tokenizer)
enc = tiktoken.get_encoding('cl100k_base')

text = "What is the capital of France?"
token_ids = enc.encode(text)
tokens    = [enc.decode([t]) for t in token_ids]

print(f'Text   : {text}')
print(f'Tokens : {tokens}')
print(f'IDs    : {token_ids}')
print(f'\nNumber of tokens: {len(token_ids)}')

Text   : What is the capital of France?
Tokens : ['What', ' is', ' the', ' capital', ' of', ' France', '?']
IDs    : [3923, 374, 279, 6864, 315, 9822, 30]

Number of tokens: 7


In [3]:
# Notice: punctuation, spaces, and subwords become separate tokens
# Let's try a trickier example
for word in ['bank', 'banking', 'unbelievable', 'GPT']:
    ids = enc.encode(word)
    parts = [enc.decode([i]) for i in ids]
    print(f'{word!r:20} → {parts} (ids: {ids})')

'bank'               → ['bank'] (ids: [17469])
'banking'            → ['bank', 'ing'] (ids: [17469, 287])
'unbelievable'       → ['un', 'belie', 'vable'] (ids: [359, 32898, 24694])
'GPT'                → ['G', 'PT'] (ids: [38, 2898])


**Key insight:** The tokenizer does not understand meaning — it just splits text into chunks it has seen frequently during BPE training. The transformer learns meaning from the token IDs.

**In nanochat's code** (`nanochat/tokenizer.py`):
```python
SPECIAL_TOKENS = [
    '<|bos|>',           # beginning of document
    '<|user_start|>',    # start of user message
    '<|assistant_start|>', # start of assistant reply
    ...
]
```
These special tokens are how nanochat formats a conversation as a flat token stream.

---
## Step 2: Token Embeddings — Token IDs → Vectors

The transformer cannot work with integers. Each token ID is looked up in an **embedding table** (a big matrix) to get a vector of floating-point numbers.

If vocab size = 32,768 and embedding dimension = 128, the table is shape `[32768, 128]`. Looking up token ID 42 returns row 42 of that table.

In [4]:
# ── Hyperparameters for our tiny educational model ─────────────────
VOCAB_SIZE = 100_000  # tiktoken cl100k vocab size
N_EMBD     = 64       # embedding dimension (nanochat uses 384-1536 depending on depth)
N_HEAD     = 4        # number of attention heads
HEAD_DIM   = N_EMBD // N_HEAD  # dimension per head = 16
N_LAYER    = 2        # number of transformer blocks
SEQ_LEN    = len(token_ids)  # our sentence length

print(f'Embedding dim   : {N_EMBD}')
print(f'Attention heads : {N_HEAD}')
print(f'Head dim        : {HEAD_DIM}')
print(f'Sequence length : {SEQ_LEN} tokens')

Embedding dim   : 64
Attention heads : 4
Head dim        : 16
Sequence length : 7 tokens


In [5]:
# ── Embedding table: shape [vocab_size, n_embd] ────────────────────
torch.manual_seed(42)
wte = nn.Embedding(VOCAB_SIZE, N_EMBD)  # 'wte' = weight token embedding (nanochat's name)

# Convert our token IDs to a tensor — shape [1, T] (batch=1, seq_len=T)
idx = torch.tensor(token_ids).unsqueeze(0)  # shape: [1, 8]
print(f'Token ID tensor shape: {idx.shape}  (batch=1, seq_len={SEQ_LEN})')

# Look up embeddings — shape [1, T, n_embd]
x = wte(idx)  # shape: [1, 8, 64]
print(f'Embedding tensor shape: {x.shape}  (batch=1, seq_len={SEQ_LEN}, n_embd={N_EMBD})')

print(f'\nEmbedding of first token "{tokens[0]}": shape {x[0,0].shape}')
print(x[0, 0][:8], '...')  # show first 8 of 64 values

Token ID tensor shape: torch.Size([1, 7])  (batch=1, seq_len=7)
Embedding tensor shape: torch.Size([1, 7, 64])  (batch=1, seq_len=7, n_embd=64)

Embedding of first token "What": shape torch.Size([64])
tensor([ 1.0441,  1.0719,  1.0265,  0.0654,  0.8637,  0.3151,  1.8440, -1.0074],
       grad_fn=<SliceBackward0>) ...


**In nanochat's `gpt.py` forward pass:**
```python
x = self.transformer.wte(idx)   # shape: [B, T, n_embd]
x = norm(x)                     # RMS normalize
```

At this point `x` is a matrix where **each row is a token, and each column is a learned feature**. The model has not learned anything yet — these are just random starting points (before training).

---
## Step 3: Q, K, V Projections

Every token's embedding is linearly projected into three vectors using three learned weight matrices:

| Vector | Formula | Meaning |
|--------|---------|--------|
| **Q** (Query) | `x @ Wq` | “What am I looking for?” |
| **K** (Key)   | `x @ Wk` | “What do I advertise I contain?” |
| **V** (Value) | `x @ Wv` | “What do I actually send if attended to?” |

In [7]:
# Three separate weight matrices — all learned during training
Wq = nn.Linear(N_EMBD, N_EMBD, bias=False)  # [64, 64]
Wk = nn.Linear(N_EMBD, N_EMBD, bias=False)  # [64, 64]
Wv = nn.Linear(N_EMBD, N_EMBD, bias=False)  # [64, 64]

# Project all tokens at once (matrix multiply over the T dimension)
Q = Wq(x)  # shape: [1, 8, 64]
K = Wk(x)  # shape: [1, 8, 64]
V = Wv(x)  # shape: [1, 8, 64]

print(f'Q shape: {Q.shape}  (batch, seq_len, n_embd)')
print(f'K shape: {K.shape}')
print(f'V shape: {V.shape}')

print(f'\nQ for token "{tokens[0]}" (first 8 values):')
print(Q[0, 0, :8].detach())
print(f'K for token "{tokens[0]}" (first 8 values):')
print(K[0, 0, :8].detach())

Q shape: torch.Size([1, 7, 64])  (batch, seq_len, n_embd)
K shape: torch.Size([1, 7, 64])
V shape: torch.Size([1, 7, 64])

Q for token "What" (first 8 values):
tensor([-0.4897, -0.1184, -0.4249, -0.1921, -0.3022, -0.0730, -0.1502, -0.1446])
K for token "What" (first 8 values):
tensor([ 0.0621, -0.4517, -0.1753, -0.8544, -0.1178, -0.0010,  0.6781, -1.0062])


**In nanochat's `gpt.py`** (inside `CausalSelfAttention.forward`):
```python
# Separate projections for Q, K, V
self.c_q = Linear(n_embd, n_head * head_dim, bias=False)
self.c_k = Linear(n_embd, n_kv_head * head_dim, bias=False)
self.c_v = Linear(n_embd, n_kv_head * head_dim, bias=False)

q = self.c_q(x).view(B, T, self.n_head, self.head_dim)
k = self.c_k(x).view(B, T, self.n_kv_head, self.head_dim)
v = self.c_v(x).view(B, T, self.n_kv_head, self.head_dim)
```

---
## Step 4: Scaled Dot-Product Attention

The full attention formula:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

- **Step 4a:** `Q @ K.T` → similarity scores between every pair of tokens
- **Step 4b:** divide by `√d` → scale down to avoid huge values (stabilizes softmax)
- **Step 4c:** `softmax` → convert scores to probabilities (sum to 1)
- **Step 4d:** multiply by `V` → weighted blend of value vectors = **context vector**

In [8]:
# Work with a single head for clarity (we'll do multi-head later)
# Reshape to split off the head dimension: [B, T, n_embd] → [B, T, n_head, head_dim]
B, T, C = x.shape

q = Q.view(B, T, N_HEAD, HEAD_DIM)  # [1, 8, 4, 16]
k = K.view(B, T, N_HEAD, HEAD_DIM)
v = V.view(B, T, N_HEAD, HEAD_DIM)

# Take just head 0 for illustration
q0 = q[:, :, 0, :]  # [1, T, 16]  — one head's queries
k0 = k[:, :, 0, :]  # [1, T, 16]
v0 = v[:, :, 0, :]  # [1, T, 16]

# ── Step 4a: Q @ K^T ──────────────────────────────────────────────
# For each token, compute its dot product against every other token
scores = q0 @ k0.transpose(-2, -1)  # [1, T, T]
print(f'Score matrix shape: {scores.shape}  (batch, T, T)')
print(f'\nRaw attention scores (token × token):')
print(scores[0].detach().round(decimals=2))

Score matrix shape: torch.Size([1, 7, 7])  (batch, T, T)

Raw attention scores (token × token):
tensor([[ 3.1500, -0.9000, -1.2200, -2.4700,  0.5000,  1.7200, -2.0100],
        [-3.2200,  2.0200, -1.1600,  1.2300, -0.6600, -0.6800, -0.3100],
        [ 0.4500, -0.5400,  0.6600,  0.4000, -0.6300, -1.3400,  1.3500],
        [-0.1400, -0.1700,  0.0600, -0.1000,  1.5300, -1.1800, -0.4900],
        [ 3.8100, -0.8100,  0.1500,  2.1500,  0.5700, -1.4300, -0.3700],
        [-0.1700, -0.1300, -0.8400, -2.5700, -0.0500, -0.5400,  2.7700],
        [-0.8400, -0.0400,  0.1900, -1.1400,  0.2800,  1.9100,  1.0000]])


In [9]:
# ── Step 4b: Scale by 1/√d ────────────────────────────────────────
# Without scaling, large embedding dims produce huge dot products
# which push softmax into saturation (gradients vanish)
d_k = HEAD_DIM  # = 16
scores_scaled = scores / math.sqrt(d_k)
print(f'After scaling by 1/√{d_k} = {1/math.sqrt(d_k):.3f}:')
print(scores_scaled[0].detach().round(decimals=2))

After scaling by 1/√16 = 0.250:
tensor([[ 0.7900, -0.2300, -0.3100, -0.6200,  0.1300,  0.4300, -0.5000],
        [-0.8100,  0.5000, -0.2900,  0.3100, -0.1600, -0.1700, -0.0800],
        [ 0.1100, -0.1300,  0.1700,  0.1000, -0.1600, -0.3400,  0.3400],
        [-0.0400, -0.0400,  0.0200, -0.0200,  0.3800, -0.2900, -0.1200],
        [ 0.9500, -0.2000,  0.0400,  0.5400,  0.1400, -0.3600, -0.0900],
        [-0.0400, -0.0300, -0.2100, -0.6400, -0.0100, -0.1300,  0.6900],
        [-0.2100, -0.0100,  0.0500, -0.2900,  0.0700,  0.4800,  0.2500]])


In [11]:
# ── Step 4c: Softmax → attention weights ──────────────────────────
attn_weights = F.softmax(scores_scaled, dim=-1)  # each row sums to 1.0
print('Attention weights (each row sums to 1.0):')
print(attn_weights[0].detach().round(decimals=3))
print(f'\nRow sums (should all be 1.0): {attn_weights[0].sum(dim=-1).detach()}')

Attention weights (each row sums to 1.0):
tensor([[0.2910, 0.1060, 0.0980, 0.0710, 0.1500, 0.2040, 0.0800],
        [0.0650, 0.2430, 0.1100, 0.1990, 0.1240, 0.1230, 0.1350],
        [0.1540, 0.1210, 0.1630, 0.1520, 0.1180, 0.0990, 0.1930],
        [0.1380, 0.1370, 0.1450, 0.1390, 0.2090, 0.1060, 0.1260],
        [0.2900, 0.0910, 0.1160, 0.1920, 0.1290, 0.0780, 0.1020],
        [0.1340, 0.1360, 0.1140, 0.0740, 0.1390, 0.1230, 0.2810],
        [0.1070, 0.1310, 0.1390, 0.0990, 0.1420, 0.2130, 0.1700]])

Row sums (should all be 1.0): tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [12]:
# ── Step 4d: Weighted sum of V → context vectors ──────────────────
context = attn_weights @ v0  # [1, T, head_dim]
print(f'Context vector shape: {context.shape}  (batch, T, head_dim)')

print(f'\nOriginal embedding of "capital" (token index 4), first 8 dims:')
print(x[0, 4, :8].detach())
print(f'\nContext vector of "capital" after attending to all tokens, first 8 dims:')
print(context[0, 4, :8].detach())
print('\n→ The context vector is different: it now contains a blend of information')
print('  from "What", "is", "the", "capital", "of", "France", etc.')

Context vector shape: torch.Size([1, 7, 16])  (batch, T, head_dim)

Original embedding of "capital" (token index 4), first 8 dims:
tensor([ 0.3701, -1.0297,  0.0140, -0.2083, -1.1175,  0.0689, -0.2811,  0.5195])

Context vector of "capital" after attending to all tokens, first 8 dims:
tensor([-0.4268,  0.1382, -0.0260,  0.3552, -0.0420, -0.0495,  0.1854, -0.2442])

→ The context vector is different: it now contains a blend of information
  from "What", "is", "the", "capital", "of", "France", etc.


---
## Step 5: Causal Masking

So far, "capital" can see "France" — but during training, that's cheating. When predicting the token *after* "capital", the model should not be able to see what comes next.

We apply a **causal mask**: set all future positions to `-inf` before softmax, so they become 0 after softmax.

In [13]:
# Build a lower-triangular mask: True where we KEEP, False where we MASK
mask = torch.tril(torch.ones(T, T)).bool()  # lower triangle = past + present
print('Causal mask (True = allowed to attend, False = blocked):')
print(mask.int())

# Show token labels on axes
print('\nTokens:', tokens)

Causal mask (True = allowed to attend, False = blocked):
tensor([[1, 0, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1]], dtype=torch.int32)

Tokens: ['What', ' is', ' the', ' capital', ' of', ' France', '?']


In [14]:
# Apply mask: replace False positions with -inf before softmax
scores_masked = scores_scaled.clone()
scores_masked = scores_masked.masked_fill(~mask, float('-inf'))

print('Masked scores (future tokens → -inf):')
print(scores_masked[0].detach().round(decimals=2))

Masked scores (future tokens → -inf):
tensor([[ 0.7900,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.8100,  0.5000,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.1100, -0.1300,  0.1700,    -inf,    -inf,    -inf,    -inf],
        [-0.0400, -0.0400,  0.0200, -0.0200,    -inf,    -inf,    -inf],
        [ 0.9500, -0.2000,  0.0400,  0.5400,  0.1400,    -inf,    -inf],
        [-0.0400, -0.0300, -0.2100, -0.6400, -0.0100, -0.1300,    -inf],
        [-0.2100, -0.0100,  0.0500, -0.2900,  0.0700,  0.4800,  0.2500]])


In [15]:
# After softmax: -inf becomes 0 — future tokens contribute nothing
attn_weights_causal = F.softmax(scores_masked, dim=-1)
print('Causal attention weights (upper triangle = 0):')
print(attn_weights_causal[0].detach().round(decimals=3))

print('\n→ "capital" (row 4) attends to tokens 0-4 only (What, is, the, capital)')
print('→ "France" (row 6) attends to tokens 0-6 — it can see "capital"')

Causal attention weights (upper triangle = 0):
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2130, 0.7870, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3530, 0.2750, 0.3720, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2460, 0.2450, 0.2600, 0.2490, 0.0000, 0.0000, 0.0000],
        [0.3540, 0.1120, 0.1420, 0.2340, 0.1580, 0.0000, 0.0000],
        [0.1870, 0.1890, 0.1580, 0.1030, 0.1930, 0.1710, 0.0000],
        [0.1070, 0.1310, 0.1390, 0.0990, 0.1420, 0.2130, 0.1700]])

→ "capital" (row 4) attends to tokens 0-4 only (What, is, the, capital)
→ "France" (row 6) attends to tokens 0-6 — it can see "capital"


**In nanochat's `gpt.py`**, causal masking is handled by Flash Attention:
```python
y = flash_attn.flash_attn_func(q, k, v, causal=True, window_size=window_size)
```
The `causal=True` flag applies the mask automatically and efficiently — no need to build a mask tensor explicitly.

---
## Step 6: Multi-Head Attention

Instead of one set of Q/K/V, the transformer runs **multiple attention heads in parallel** — each with its own Wq, Wk, Wv. Each head can learn to attend to different relationships.

For example:
- Head 0 might learn syntactic relationships (subject → verb)
- Head 1 might learn semantic relationships ("capital" → country names)
- Head 2 might learn coreference (pronouns → their referents)

All heads' outputs are concatenated and projected back to `n_embd`.

In [16]:
def causal_self_attention(x, Wq, Wk, Wv, Wo):
    """Full multi-head causal self-attention."""
    B, T, C = x.shape
    
    # Project to Q, K, V and split into heads: [B, T, C] → [B, T, n_head, head_dim]
    q = Wq(x).view(B, T, N_HEAD, HEAD_DIM)
    k = Wk(x).view(B, T, N_HEAD, HEAD_DIM)
    v = Wv(x).view(B, T, N_HEAD, HEAD_DIM)
    
    # Transpose for attention: [B, n_head, T, head_dim]
    q = q.transpose(1, 2)
    k = k.transpose(1, 2)
    v = v.transpose(1, 2)
    
    # Scaled dot-product attention for all heads at once
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(HEAD_DIM)  # [B, n_head, T, T]
    
    # Causal mask
    mask = torch.tril(torch.ones(T, T, device=x.device)).bool()
    scores = scores.masked_fill(~mask, float('-inf'))
    
    # Softmax + weighted sum
    attn = F.softmax(scores, dim=-1)  # [B, n_head, T, T]
    out  = attn @ v                   # [B, n_head, T, head_dim]
    
    # Concatenate heads: [B, n_head, T, head_dim] → [B, T, n_embd]
    out = out.transpose(1, 2).contiguous().view(B, T, C)
    
    # Final output projection
    out = Wo(out)  # [B, T, n_embd]
    return out, attn

Wo = nn.Linear(N_EMBD, N_EMBD, bias=False)  # output projection
attn_out, attn_weights_all = causal_self_attention(x.detach(), Wq, Wk, Wv, Wo)

print(f'Multi-head attention output shape: {attn_out.shape}')
print(f'Attention weights shape: {attn_weights_all.shape}  (batch, n_head, T, T)')
print(f'\nEach of the {N_HEAD} heads has its own T×T attention pattern:')
for h in range(N_HEAD):
    print(f'  Head {h} — attention weights for token "capital" (row 4):')
    print(f'  {attn_weights_all[0, h, 4].detach().round(decimals=3).tolist()}')

Multi-head attention output shape: torch.Size([1, 7, 64])
Attention weights shape: torch.Size([1, 4, 7, 7])  (batch, n_head, T, T)

Each of the 4 heads has its own T×T attention pattern:
  Head 0 — attention weights for token "capital" (row 4):
  [0.3540000021457672, 0.1120000034570694, 0.1420000046491623, 0.23399999737739563, 0.15800000727176666, 0.0, 0.0]
  Head 1 — attention weights for token "capital" (row 4):
  [0.16699999570846558, 0.22300000488758087, 0.16500000655651093, 0.24300000071525574, 0.20100000500679016, 0.0, 0.0]
  Head 2 — attention weights for token "capital" (row 4):
  [0.21199999749660492, 0.10300000011920929, 0.210999995470047, 0.12800000607967377, 0.34599998593330383, 0.0, 0.0]
  Head 3 — attention weights for token "capital" (row 4):
  [0.22699999809265137, 0.17900000512599945, 0.22499999403953552, 0.19300000369548798, 0.17599999904632568, 0.0, 0.0]


---
## Step 7: The MLP Block

After attention, each token's context vector passes through a **feed-forward MLP** (multi-layer perceptron). This applies the same transformation to each token independently.

The MLP expands to `4 × n_embd`, applies a non-linearity, then projects back down. This is where the model stores "factual" knowledge — patterns that don't depend on context (e.g., "Paris is the capital of France").

In [17]:
class MLP(nn.Module):
    """Feed-forward block — identical to nanochat's MLP in gpt.py."""
    def __init__(self, n_embd):
        super().__init__()
        self.fc1  = nn.Linear(n_embd, 4 * n_embd, bias=False)  # expand
        self.proj = nn.Linear(4 * n_embd, n_embd, bias=False)  # contract

    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x).square()  # relu² activation — nanochat's choice
        x = self.proj(x)
        return x

mlp = MLP(N_EMBD)
mlp_out = mlp(attn_out.detach())

print(f'MLP input  shape: {attn_out.shape}')
print(f'MLP output shape: {mlp_out.shape}  (same shape — token representations updated)')
print(f'\nInside the MLP, the hidden layer is 4×{N_EMBD} = {4*N_EMBD} wide')

MLP input  shape: torch.Size([1, 7, 64])
MLP output shape: torch.Size([1, 7, 64])  (same shape — token representations updated)

Inside the MLP, the hidden layer is 4×64 = 256 wide


**nanochat's `gpt.py`:**
```python
class MLP(nn.Module):
    def forward(self, x):
        x = self.c_fc(x)
        x = F.relu(x).square()   # relu² — slightly better than plain relu
        x = self.c_proj(x)
        return x
```

**Attention vs. MLP — the key distinction:**
- **Attention** gathers information *across* tokens (token-to-token communication)
- **MLP** processes each token *independently* (token-level feature transformation)

---
## Step 8: Residual Connections & Layer Norm

Two crucial stabilization tricks used in every transformer:

**Residual connection:** Add the input back after each sub-block.
```
x = x + attention(x)
x = x + mlp(x)
```
This gives gradients a "highway" to flow backwards during training without vanishing.

**Layer Norm (RMSNorm in nanochat):** Normalize the vector before each sub-block so values don't explode.

In [18]:
def rms_norm(x):
    """RMS Normalization — same as nanochat's norm() in gpt.py."""
    return F.rms_norm(x, (x.size(-1),))

# Without residual: the original token information is overwritten
x_no_residual = mlp_out.detach()  # pure MLP output

# With residual: the original embedding is preserved and enriched
x_with_residual = x.detach() + attn_out.detach()  # skip connection over attention
x_with_residual = x_with_residual + mlp(rms_norm(x_with_residual)).detach()

print('Residual connection preserves information through deep networks.')
print(f'\nOriginal embedding norm : {x[0,4].norm().item():.3f}')
print(f'After attention + residual norm: {(x + attn_out)[0,4].detach().norm().item():.3f}')
print('\n→ The norm stays stable. Without residuals, it would drift and vanish.')

Residual connection preserves information through deep networks.

Original embedding norm : 9.001
After attention + residual norm: 8.909

→ The norm stays stable. Without residuals, it would drift and vanish.


**In nanochat's `gpt.py`** (the `Block.forward` method):
```python
class Block(nn.Module):
    def forward(self, x, ...):
        x = x + self.attn(norm(x), ...)  # attention with residual
        x = x + self.mlp(norm(x))        # MLP with residual
        return x
```
Note: norm is applied *before* each sub-block ("pre-norm"), which is more stable than post-norm.

---
## Step 9: A Full Transformer Block

Putting it all together: one transformer block = attention + MLP, each with pre-norm and residual.

In [19]:
class TransformerBlock(nn.Module):
    """One transformer block — mirrors nanochat's Block class in gpt.py."""
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_dim = n_embd // n_head
        self.Wq = nn.Linear(n_embd, n_embd, bias=False)
        self.Wk = nn.Linear(n_embd, n_embd, bias=False)
        self.Wv = nn.Linear(n_embd, n_embd, bias=False)
        self.Wo = nn.Linear(n_embd, n_embd, bias=False)
        self.mlp = MLP(n_embd)
        self.n_head   = n_head
        self.head_dim = head_dim

    def attention(self, x):
        B, T, C = x.shape
        q = self.Wq(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.Wk(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = self.Wv(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask   = torch.tril(torch.ones(T, T, device=x.device)).bool()
        scores = scores.masked_fill(~mask, float('-inf'))
        attn   = F.softmax(scores, dim=-1)
        out    = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.Wo(out)

    def forward(self, x):
        # Pre-norm → attention → residual
        x = x + self.attention(rms_norm(x))
        # Pre-norm → MLP → residual
        x = x + self.mlp(rms_norm(x))
        return x

block = TransformerBlock(N_EMBD, N_HEAD)
x_after_block = block(x.detach())

print(f'Input  shape: {x.shape}')
print(f'Output shape: {x_after_block.shape}  (same — blocks preserve shape)')
print('\nOne block has:')
print(f'  Wq, Wk, Wv, Wo : 4 × {N_EMBD}×{N_EMBD} = {4*N_EMBD*N_EMBD:,} params')
print(f'  MLP fc1         : {N_EMBD}×{4*N_EMBD} = {N_EMBD*4*N_EMBD:,} params')
print(f'  MLP proj        : {4*N_EMBD}×{N_EMBD} = {4*N_EMBD*N_EMBD:,} params')
total = 4*N_EMBD*N_EMBD + N_EMBD*4*N_EMBD + 4*N_EMBD*N_EMBD
print(f'  Total per block : {total:,} params')

Input  shape: torch.Size([1, 7, 64])
Output shape: torch.Size([1, 7, 64])  (same — blocks preserve shape)

One block has:
  Wq, Wk, Wv, Wo : 4 × 64×64 = 16,384 params
  MLP fc1         : 64×256 = 16,384 params
  MLP proj        : 256×64 = 16,384 params
  Total per block : 49,152 params


---
## Step 10: Stacking Layers

A GPT model is just N transformer blocks stacked sequentially. Each block takes the previous block's output as input. This is the `--depth` parameter in nanochat.

```
embeddings
    ↓
 Block 0   ← layer 0: learns low-level patterns (syntax, word order)
    ↓
 Block 1   ← layer 1: builds on block 0's representations
    ↓
  ...       ← deeper layers: increasingly abstract, semantic concepts
    ↓
 Block N-1 ← final layer: high-level meaning, ready to predict next token
```

In [20]:
# Stack N_LAYER blocks
blocks = nn.ModuleList([TransformerBlock(N_EMBD, N_HEAD) for _ in range(N_LAYER)])

# Forward pass through all layers
h = x.detach().clone()
print(f'Layer 0 (embeddings) — "capital" vector norm: {h[0,4].norm().item():.3f}')
for i, block in enumerate(blocks):
    h = block(h)
    print(f'Layer {i+1} (after block {i}) — "capital" vector norm: {h[0,4].norm().item():.3f}')

print(f'\nFinal output shape: {h.shape}  (batch, seq_len, n_embd)')
print('The vector for each token now encodes its meaning in context.')

Layer 0 (embeddings) — "capital" vector norm: 9.001
Layer 1 (after block 0) — "capital" vector norm: 8.979
Layer 2 (after block 1) — "capital" vector norm: 9.094

Final output shape: torch.Size([1, 7, 64])  (batch, seq_len, n_embd)
The vector for each token now encodes its meaning in context.


**In nanochat's `gpt.py` forward:**
```python
# Forward the trunk of the Transformer
for i, block in enumerate(self.transformer.h):
    x = self.resid_lambdas[i] * x + self.x0_lambdas[i] * x0
    ve = self.value_embeds[str(i)](idx)  # value embeddings (ResFormer)
    x = block(x, ve, cos_sin, self.window_sizes[i], kv_cache)
```
nanochat adds a few extras: per-layer residual scaling (`resid_lambdas`), input blending (`x0_lambdas`), and Value Embeddings — but the core is the same loop.

---
## Step 11: Next Token Prediction

After all layers, the final token's context vector is fed through a **language model head** — a linear layer that maps `n_embd → vocab_size`. This produces **logits**: one score per token in the vocabulary.

The token with the highest logit (or a sampled token) is the model's prediction for what comes next.

In [21]:
# The language model head: n_embd → vocab_size
lm_head = nn.Linear(N_EMBD, VOCAB_SIZE, bias=False)

# Apply final norm, then project to vocabulary
h_normed = rms_norm(h)           # [1, T, n_embd]
logits   = lm_head(h_normed)     # [1, T, vocab_size]

print(f'Logits shape: {logits.shape}  (batch, seq_len, vocab_size)')
print(f'\nFor next-token prediction we only need the LAST token\'s logits:')

last_logits = logits[0, -1, :]   # [vocab_size] — predictions after "France?"
print(f'Last logits shape: {last_logits.shape}')

# Convert logits to probabilities
probs = F.softmax(last_logits, dim=-1)
top5  = probs.topk(5)

print(f'\nTop-5 predicted next tokens (untrained model — random probabilities):')
for prob, idx_val in zip(top5.values.detach(), top5.indices.detach()):
    token_str = enc.decode([idx_val.item()])
    print(f'  {token_str!r:20} (id={idx_val.item():6d}) → prob={prob.item():.4f}')

Logits shape: torch.Size([1, 7, 100000])  (batch, seq_len, vocab_size)

For next-token prediction we only need the LAST token's logits:
Last logits shape: torch.Size([100000])

Top-5 predicted next tokens (untrained model — random probabilities):
  '(SP'                (id= 87603) → prob=0.0001
  ' Jain'              (id= 96217) → prob=0.0001
  ' Wo'                (id= 28357) → prob=0.0001
  'arra'               (id= 80130) → prob=0.0001
  '.$$'                (id= 77566) → prob=0.0001


The probabilities are random because we have an **untrained model**. During training, the model adjusts all the weight matrices (Wq, Wk, Wv, Wo, MLP weights, lm_head) using gradient descent to make the correct next token have a higher probability.

**In nanochat's `gpt.py`:**
```python
# Forward the lm_head (compute logits)
softcap = 15   # smoothly cap logits to [-15, 15] for stability
logits = self.lm_head(x)    # [B, T, vocab_size]
logits = softcap * torch.tanh(logits / softcap)
logits = logits[:, :, :self.config.vocab_size]  # crop padding

# Compute cross-entropy loss against target tokens
loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
```

---
## Step 12: The Full Picture

Let's trace the complete journey of our sentence through the transformer:

In [22]:
class TinyGPT(nn.Module):
    """Minimal GPT — the same architecture as nanochat, simplified for learning."""
    def __init__(self, vocab_size, n_embd, n_head, n_layer):
        super().__init__()
        self.wte     = nn.Embedding(vocab_size, n_embd)        # token embeddings
        self.blocks  = nn.ModuleList([
            TransformerBlock(n_embd, n_head) for _ in range(n_layer)
        ])
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx):
        # Step 1: Embed tokens
        x = self.wte(idx)                    # [B, T, n_embd]
        x = rms_norm(x)

        # Step 2: Pass through transformer blocks
        for block in self.blocks:
            x = block(x)                     # [B, T, n_embd]

        # Step 3: Predict next token
        x = rms_norm(x)
        logits = self.lm_head(x)             # [B, T, vocab_size]
        return logits

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=20, temperature=1.0, top_k=50):
        """Autoregressive generation: repeatedly predict and append next token."""
        for _ in range(max_new_tokens):
            logits  = self.forward(idx)            # [1, T, vocab_size]
            logits  = logits[:, -1, :] / temperature  # take last token's logits
            # Top-k sampling
            v, _    = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = float('-inf')
            probs   = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)  # sample
            idx     = torch.cat([idx, next_id], dim=1)         # append
        return idx

# Build the model
model = TinyGPT(VOCAB_SIZE, N_EMBD, N_HEAD, N_LAYER)
total_params = sum(p.numel() for p in model.parameters())
print(f'TinyGPT parameters: {total_params:,}')
print(f'  Embedding table : {VOCAB_SIZE * N_EMBD:,}')
print(f'  Per block       : ~{4*N_EMBD*N_EMBD + 2*N_EMBD*4*N_EMBD:,}  × {N_LAYER} blocks')
print(f'  LM head         : {VOCAB_SIZE * N_EMBD:,}')

TinyGPT parameters: 12,898,304
  Embedding table : 6,400,000
  Per block       : ~49,152  × 2 blocks
  LM head         : 6,400,000


In [23]:
# Run a full forward pass
idx = torch.tensor(token_ids).unsqueeze(0)  # [1, T]
logits = model(idx)
print(f'Input  shape: {idx.shape}')
print(f'Logits shape: {logits.shape}  (one set of logits per input token)')

# Generate some tokens (will be random/nonsense — model is untrained)
print(f'\nInput sentence: "{text}"')
generated = model.generate(idx, max_new_tokens=10)
new_tokens = enc.decode(generated[0][len(token_ids):].tolist())
print(f'Generated (untrained, random): "{new_tokens}"')
print('\n→ Gibberish because the model has not been trained yet.')
print('→ Training adjusts the weights so the model learns to produce sensible continuations.')

Input  shape: torch.Size([1, 7])
Logits shape: torch.Size([1, 7, 100000])  (one set of logits per input token)

Input sentence: "What is the capital of France?"
Generated (untrained, random): ""/>

 ${
 inflict Mexico是 buffer eclJECTED textarea luggage"

→ Gibberish because the model has not been trained yet.
→ Training adjusts the weights so the model learns to produce sensible continuations.


---
## Step 13: Running nanochat's Actual Model on Your Mac

Now let's actually run nanochat's real code. This uses the production model from `nanochat/gpt.py` with its extra features (rotary embeddings, QK norm, value embeddings, etc.).

**The Mac training run takes ~90-120 min on M1 Pro.** Run these in your terminal:

In [24]:
# ── Verify the nanochat model can be imported and instantiated ─────
import sys
import os

# Make sure we can import nanochat from the project root
project_root = os.path.dirname(os.path.abspath('.'))
if project_root not in sys.path:
    sys.path.insert(0, os.getcwd())

from nanochat.gpt import GPT, GPTConfig

# Build the same small model used in runcpu.sh (depth=6)
config = GPTConfig(
    sequence_len = 512,
    vocab_size   = 32768,
    n_layer      = 6,
    n_head       = 6,
    n_kv_head    = 6,
    n_embd       = 384,   # 6 layers × 64 aspect ratio
    window_pattern = 'L',  # all layers use full context (simpler)
)

with torch.device('meta'):  # meta device: allocate shape only, no memory
    nanochat_model = GPT(config)

total = sum(p.numel() for p in nanochat_model.parameters())
print(f'nanochat GPT config: depth=6, n_embd=384, n_head=6')
print(f'Total parameters: {total/1e6:.1f}M')
print(f'\nCompare to our TinyGPT: {total_params/1e3:.0f}K  (our toy model)')
print(f'nanochat is {total/total_params:.0f}× larger')

nanochat GPT config: depth=6, n_embd=384, n_head=6
Total parameters: 73.5M

Compare to our TinyGPT: 12898K  (our toy model)
nanochat is 6× larger


### How to train nanochat on your Mac

Open a terminal in the nanochat directory and run:

```bash
# Install CPU dependencies (MPS will be auto-detected)
uv sync --extra cpu
source .venv/bin/activate

# Run the full Mac pipeline (~90-120 min on M1 Pro)
bash runs/runcpu.sh
```

This will:
1. Download ~2B chars of text data
2. Train a custom BPE tokenizer (vocab=32K) — ~1-2 min
3. Pretrain a 6-layer model for 5,000 iterations — ~60-90 min
4. Fine-tune on conversation data (SFT) — ~20-30 min

After training, chat with your model:

```bash
# CLI chat
python -m scripts.chat_cli -p "What is the capital of France?"

# Web UI (ChatGPT-style)
python -m scripts.chat_web
# then open http://localhost:8000
```

---
## Appendix: nanochat's Extra Features

nanochat's production model (`nanochat/gpt.py`) adds several improvements over our simplified version:

| Feature | Our TinyGPT | nanochat |
|---------|------------|----------|
| Positional encoding | ❌ none | ✅ Rotary embeddings (RoPE) |
| Attention stability | basic softmax | ✅ QK norm |
| Memory efficiency | all full context | ✅ Sliding window attention |
| Inference speed | recompute all tokens | ✅ KV cache |
| Multi-GPU training | ❌ | ✅ DDP via torchrun |
| Precision | float32 | ✅ bfloat16 / FP8 |
| Optimizer | Adam | ✅ Muon + AdamW |
| Value enrichment | standard V | ✅ Value Embeddings (ResFormer) |

Each of these is an engineering improvement, but the **core architecture is identical** to what you built in this notebook.

---
## Summary

The transformer in one sentence: **tokens become vectors, vectors ask questions of each other via attention, their answers are blended and refined through MLP layers, and the final vector predicts the next token.**

The full flow:
```
text
 ↓  tokenize
token IDs  [42, 318, 262, ...]
 ↓  embedding table (wte)
vectors    [[0.1, -0.3, ...], ...]    ← context-free
 ↓  block 0: attention + MLP
vectors    [[0.4, 0.2,  ...], ...]    ← slightly context-aware
 ↓  block 1 ... block N
vectors    [[0.9, -0.1, ...], ...]    ← deeply context-aware
 ↓  lm_head
logits     [0.01, 0.3, 0.002, ...]   ← score for every vocab token
 ↓  softmax + sample
next token  "Paris"
```

---
---
# Part 2: Train a Real GPT on Tiny Shakespeare

This section fulfils the full assignment:
- A complete GPT model with **token + positional embeddings**
- A **training pipeline** with minibatch sampling, AdamW optimiser, and validation
- **Loss curves** (train and val loss vs iteration)
- **Generated text samples** from the trained model
- A **discussion** of architecture choices

The model is small enough to train on an M4 Mac in **~5 minutes**.

---
## Architecture Note

| Hyperparameter | Value | Reason |
|---|---|---|
| Dataset | Tiny Shakespeare (~1M chars) | Small, well-known, clearly structured |
| Tokenisation | Character-level (65 tokens) | Tiny vocab → tiny embedding table, fast to train |
| `n_layer` | 4 | Deep enough to learn multi-step patterns, light enough for CPU/MPS |
| `n_embd` | 128 | Embedding dimension; wide enough to store character context |
| `n_head` | 4 | 4 heads × 32 head-dim = 128; each head attends to different patterns |
| `block_size` | 128 | Context window; 128 chars ≈ 3–4 lines of Shakespeare |
| `batch_size` | 64 | Minibatch size; fits comfortably in M4 unified memory |
| `max_iters` | 5000 | ~5 min on M4 MPS; loss clearly converges |
| Optimiser | AdamW, lr=3e-4 | Standard choice; weight decay prevents overfitting |

Total parameters: **~820K** — tiny by modern standards, but sufficient to learn Shakespeare's style.

---
## Step 14: Dataset — Tiny Shakespeare

We download ~1 million characters of Shakespeare plays and build a **character-level tokeniser** — the simplest possible tokeniser: each unique character gets an integer ID.

In [ ]:
import urllib.request, os, torch, torch.nn as nn, torch.nn.functional as F, math, time

# ── Download Tiny Shakespeare ──────────────────────────────────────
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
data_path = 'tinyshakespeare.txt'
if not os.path.exists(data_path):
    print("Downloading Tiny Shakespeare...")
    urllib.request.urlretrieve(url, data_path)
    print("Done.")
with open(data_path, 'r') as f:
    raw_text = f.read()

print(f"Total characters : {len(raw_text):,}")
print(f"\nFirst 300 chars:\n{raw_text[:300]}")

In [ ]:
# ── Character-level tokeniser ─────────────────────────────────────
chars   = sorted(set(raw_text))
CHAR_VOCAB_SIZE = len(chars)
stoi    = {c: i for i, c in enumerate(chars)}
itos    = {i: c for i, c in enumerate(chars)}
encode  = lambda s: [stoi[c] for c in s]
decode  = lambda l: ''.join([itos[i] for i in l])

print(f"Vocabulary size  : {CHAR_VOCAB_SIZE} unique characters")
print(f"Vocabulary       : {repr(''.join(chars))}")
print(f"\nExample encoding of 'Hello'  : {encode('Hello')}")
print(f"Example decoding of {encode('Hello')} : {repr(decode(encode('Hello')))}")

# ── Train / validation split (90% / 10%) ─────────────────────────
data       = torch.tensor(encode(raw_text), dtype=torch.long)
n          = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

print(f"\nTotal tokens  : {len(data):,}")
print(f"Train tokens  : {len(train_data):,}  (90%)")
print(f"Val tokens    : {len(val_data):,}   (10%)")

---
## Step 15: Minibatch Sampling

For each training step we randomly sample a batch of `(input, target)` sequences from the corpus.
- **input**  `x`: characters at positions `i` to `i+block_size`
- **target** `y`: characters at positions `i+1` to `i+block_size+1`  (shifted by 1)

The model's job: given `x`, predict `y` at every position.

In [ ]:
# ── Device ────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f"Training device: {DEVICE}")

# ── Hyperparameters ───────────────────────────────────────────────
BLOCK_SIZE  = 128
BATCH_SIZE  = 64
TRAIN_DATA  = train_data.to(DEVICE)
VAL_DATA    = val_data.to(DEVICE)

def get_batch(split):
    """Sample a random minibatch of (input, target) pairs."""
    data = TRAIN_DATA if split == 'train' else VAL_DATA
    ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x  = torch.stack([data[i : i + BLOCK_SIZE]     for i in ix])
    y  = torch.stack([data[i+1 : i + BLOCK_SIZE+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print(f"\nBatch input  shape: {xb.shape}  (batch_size=64, block_size=128)")
print(f"Batch target shape: {yb.shape}")
print(f"\nFirst sequence input  (decoded): {repr(decode(xb[0].cpu().tolist()))[:80]}...")
print(f"First sequence target (decoded): {repr(decode(yb[0].cpu().tolist()))[:80]}...")
print("\n→ Target is input shifted by 1: the model predicts each next character.")

---
## Step 16: GPT Model with Positional Embeddings

The key addition over Part 1's `TinyGPT`: a **learned positional embedding table** `wpe`.

```
x = wte(token_ids)          # token embeddings:    [B, T, n_embd]
x = x + wpe(positions)      # positional embeddings: [B, T, n_embd]
```

Without positional embeddings, the model has no way to know whether "the" appears at position 0 or position 50 — attention is permutation-invariant.

In [ ]:
# ── Re-use building blocks from Part 1 ───────────────────────────
# (MLP, TransformerBlock, rms_norm are already defined above)

class GPTShakespeare(nn.Module):
    """
    Minimal GPT for Tiny Shakespeare.
    Architecture: Token Embedding + Positional Embedding
                  → L × TransformerBlock → LayerNorm → Linear LM Head
    """
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size):
        super().__init__()
        self.block_size = block_size
        self.wte     = nn.Embedding(vocab_size, n_embd)   # token embeddings
        self.wpe     = nn.Embedding(block_size, n_embd)   # positional embeddings ← NEW
        self.blocks  = nn.ModuleList([
            TransformerBlock(n_embd, n_head) for _ in range(n_layer)
        ])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size

        pos     = torch.arange(T, device=idx.device)
        tok_emb = self.wte(idx)                    # [B, T, n_embd]
        pos_emb = self.wpe(pos)                    # [T,    n_embd]
        x = tok_emb + pos_emb                      # [B, T, n_embd]

        for block in self.blocks:
            x = block(x)

        x      = self.ln_f(x)
        logits = self.lm_head(x)                   # [B, T, vocab_size]

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=200, temperature=0.8, top_k=40):
        """Autoregressive generation — append one token at a time."""
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits     = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs   = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx     = torch.cat([idx, next_id], dim=1)
        self.train()
        return idx

# ── Instantiate ───────────────────────────────────────────────────
torch.manual_seed(42)
gpt = GPTShakespeare(
    vocab_size  = CHAR_VOCAB_SIZE,
    n_embd      = 128,
    n_head      = 4,
    n_layer     = 4,
    block_size  = BLOCK_SIZE,
).to(DEVICE)

n_params = sum(p.numel() for p in gpt.parameters())
print(f"GPTShakespeare — total parameters: {n_params:,}")
print(f"  wte  (token embed)    : {CHAR_VOCAB_SIZE * 128:,}")
print(f"  wpe  (position embed) : {BLOCK_SIZE * 128:,}  ← new vs Part 1")
print(f"  4 blocks              : {(4*128*128 + 2*128*512) * 4:,}")
print(f"  lm_head               : {CHAR_VOCAB_SIZE * 128:,}")
print(f"\nDevice: {next(gpt.parameters()).device}")

---
## Step 17: Training Loop

A standard training loop with:
1. **Forward pass** — compute logits and cross-entropy loss
2. **Backward pass** — compute gradients
3. **Optimiser step** — update weights
4. **Periodic validation** — estimate val loss every `eval_interval` steps without gradient computation

In [ ]:
import matplotlib.pyplot as plt

MAX_ITERS     = 5000
EVAL_INTERVAL = 250
EVAL_ITERS    = 50
LR            = 3e-4

optimiser = torch.optim.AdamW(gpt.parameters(), lr=LR, weight_decay=1e-1)

@torch.no_grad()
def estimate_loss():
    gpt.eval()
    out = {}
    for split in ('train', 'val'):
        losses = []
        for _ in range(EVAL_ITERS):
            xb, yb = get_batch(split)
            _, loss = gpt(xb, yb)
            losses.append(loss.item())
        out[split] = sum(losses) / len(losses)
    gpt.train()
    return out

train_losses, val_losses, eval_steps = [], [], []

print(f"Training GPTShakespeare for {MAX_ITERS} iterations on {DEVICE}...")
print(f"Evaluating every {EVAL_INTERVAL} steps.\n")

t0 = time.time()
for step in range(MAX_ITERS + 1):
    if step % EVAL_INTERVAL == 0:
        losses  = estimate_loss()
        elapsed = time.time() - t0
        eta     = (elapsed / max(step, 1)) * (MAX_ITERS - step)
        print(f"step {step:5d}/{MAX_ITERS} | "
              f"train loss {losses['train']:.4f} | "
              f"val loss {losses['val']:.4f} | "
              f"elapsed {elapsed/60:.1f}m | eta {eta/60:.1f}m")
        train_losses.append(losses['train'])
        val_losses.append(losses['val'])
        eval_steps.append(step)
    if step == MAX_ITERS:
        break
    xb, yb       = get_batch('train')
    logits, loss = gpt(xb, yb)
    optimiser.zero_grad(set_to_none=True)
    loss.backward()
    optimiser.step()

total_time = time.time() - t0
print(f"\nTraining complete in {total_time/60:.1f} minutes.")
print(f"Final train loss: {train_losses[-1]:.4f}")
print(f"Final val   loss: {val_losses[-1]:.4f}")

---
## Step 18: Loss Curves

Plot of training and validation loss vs. iteration.
The two curves should track closely — a sign the small model is not severely overfitting.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(eval_steps, train_losses, label='Train loss', linewidth=2, color='#7c6af7')
ax.plot(eval_steps, val_losses,   label='Val loss',   linewidth=2, color='#f59e0b', linestyle='--')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Cross-entropy loss', fontsize=12)
ax.set_title('GPTShakespeare — Training on Tiny Shakespeare (char-level)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.annotate(f'Final train: {train_losses[-1]:.3f}',
            xy=(eval_steps[-1], train_losses[-1]),
            xytext=(-80, 10), textcoords='offset points', fontsize=9, color='#7c6af7')
ax.annotate(f'Final val: {val_losses[-1]:.3f}',
            xy=(eval_steps[-1], val_losses[-1]),
            xytext=(-80, -18), textcoords='offset points', fontsize=9, color='#f59e0b')
plt.tight_layout()
plt.savefig('loss_curves.png', dpi=120)
plt.show()
print("Loss curve saved to loss_curves.png")

---
## Step 19: Text Generation — Qualitative Evaluation

Sample text from the trained model.
We prime it with a short context and let it generate 500 characters autoregressively.

In [ ]:
def sample(prompt, max_new_tokens=500, temperature=0.8, top_k=40):
    """Generate text from a string prompt."""
    ids    = torch.tensor(encode(prompt), dtype=torch.long).unsqueeze(0).to(DEVICE)
    output = gpt.generate(ids, max_new_tokens=max_new_tokens,
                          temperature=temperature, top_k=top_k)
    return decode(output[0].cpu().tolist())

print("=" * 70)
print("Sample 1 — prompt: 'ROMEO:'")
print("=" * 70)
print(sample("ROMEO:"))

print()
print("=" * 70)
print("Sample 2 — prompt: 'To be or not to be'")
print("=" * 70)
print(sample("To be or not to be"))

print()
print("=" * 70)
print("Sample 3 — prompt: 'First Citizen:'")
print("=" * 70)
print(sample("First Citizen:"))

---
## Step 20: What Affects Performance?

### 1. Model size (`n_embd`, `n_layer`, `n_head`)
Larger models learn richer representations but take longer to train. Our 4-layer, 128-dim model (~820K params) generates syntactically plausible Shakespeare but makes semantic errors. A 6-layer, 256-dim model would improve noticeably.

### 2. Context length (`block_size`)
Longer context lets the model see more history — important for Shakespeare where a character's line depends on what was said many turns ago. Increasing `block_size` from 128 to 256 typically improves coherence but quadruples the attention computation.

### 3. Training time / iterations
More iterations = lower loss = better text. At 5,000 iterations the model has seen ~5,000 × 64 × 128 = **~41M tokens** (about 41 passes over the 1M character dataset). Training longer would continue to improve quality.

### 4. Tokenisation granularity
Character-level tokenisation (65 tokens) is simple but inefficient — the model must learn to spell words from scratch. A BPE tokeniser (like nanochat's 32K vocab) encodes words as whole units, letting the model focus on higher-level patterns. For this dataset, character-level is sufficient and faster to train.

### 5. Positional embeddings
Without `wpe`, the model has no notion of position — "the king said" and "said the king" would look identical after summing embeddings. Adding learned positional embeddings (one vector per position 0…block_size-1) lets the model distinguish word order, which is critical for language.

### 6. Learning rate
AdamW at lr=3e-4 is a reliable default for small transformers. Too high → loss spikes; too low → slow convergence. A learning rate schedule (warmup + cosine decay, as used by nanochat) would further improve results.

---
*This notebook implements the same core architecture as nanochat's `gpt.py`, stripped to its essentials. The cloud-trained nanochat model (1.38B params, 5568 steps on 8×H100) is the same design scaled up by ~1700×.*